# Step 1: Import Libraries

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Step 2: Load Dataset

In [28]:
train = pd.read_csv(r"C:\Users\Abhay\OneDrive\Desktop\AI-Fertilizer-Recommendation\data\train.csv")

train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


# Step 3: Drop Unnecessary Columns

In [29]:
train = train.drop(columns=["id"])

# Step 4: Separate Features and Target

In [30]:
X = train.drop("Fertilizer Name", axis=1)

y = train["Fertilizer Name"]
print(X.head())
print(y.head())

   Temparature  Humidity  Moisture Soil Type  Crop Type  Nitrogen  Potassium  \
0           37        70        36    Clayey  Sugarcane        36          4   
1           27        69        65     Sandy    Millets        30          6   
2           29        63        32     Sandy    Millets        24         12   
3           35        62        54     Sandy     Barley        39         12   
4           35        58        43       Red      Paddy        37          2   

   Phosphorous  
0            5  
1           18  
2           16  
3            4  
4           16  
0       28-28
1       28-28
2    17-17-17
3    10-26-26
4         DAP
Name: Fertilizer Name, dtype: str


# Step 5: Identify Categorical Columns

In [31]:
X.select_dtypes(include="object").columns

C:\Users\Abhay\AppData\Local\Temp\ipykernel_25084\2213039483.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include="object").columns


Index(['Soil Type', 'Crop Type'], dtype='str')

# Step 6: Encode Categorical Features

In [32]:
soil_encoder = LabelEncoder()
crop_encoder = LabelEncoder()

X["Soil Type"] = soil_encoder.fit_transform(X["Soil Type"])
X["Crop Type"] = crop_encoder.fit_transform(X["Crop Type"])
X.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,37,70,36,1,8,36,4,5
1,27,69,65,4,4,30,6,18
2,29,63,32,4,4,24,12,16
3,35,62,54,4,0,39,12,4
4,35,58,43,3,6,37,2,16


# Step 7: Encode Target Variable

In [33]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

In [34]:
mapping = dict(zip(target_encoder.classes_,
                   target_encoder.transform(target_encoder.classes_)))

print(mapping)

{'10-26-26': np.int64(0), '14-35-14': np.int64(1), '17-17-17': np.int64(2), '20-20': np.int64(3), '28-28': np.int64(4), 'DAP': np.int64(5), 'Urea': np.int64(6)}


# Step 8: Check Data Types

In [35]:
X.dtypes

Temparature    int64
Humidity       int64
Moisture       int64
Soil Type      int64
Crop Type      int64
Nitrogen       int64
Potassium      int64
Phosphorous    int64
dtype: object

# Step 9: Train-Test Split

In [36]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape)
print(X_valid.shape)

(600000, 8)
(150000, 8)


# Step 10: Save Encoders

In [37]:
import joblib
from pathlib import Path

root = Path.cwd()
for parent in [root, *root.parents]:
    if (parent / "backend").exists() and (parent / "data").exists():
        root = parent
        break

saved_model_dir = root / "backend" / "saved_models"
saved_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(soil_encoder, saved_model_dir / "soil_encoder.pkl")
joblib.dump(crop_encoder, saved_model_dir / "crop_encoder.pkl")
joblib.dump(target_encoder, saved_model_dir / "target_encoder.pkl")

['c:\\Users\\Abhay\\OneDrive\\Desktop\\AI-Fertilizer-Recommendation\\backend\\saved_models\\target_encoder.pkl']

# Step 11: Save Processed Data (Optional)

In [38]:
data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(data_dir / "X_train.csv", index=False)
X_valid.to_csv(data_dir / "X_valid.csv", index=False)

pd.DataFrame(y_valid).to_csv(data_dir / "y_valid.csv", index=False)
pd.DataFrame(y_train).to_csv(data_dir / "y_train.csv", index=False)

# Step 12: Verify Everything

In [39]:
print("Training Shape:", X_train.shape)
print("Validation Shape:", X_valid.shape)

print("Unique Fertilizers:", len(np.unique(y)))

print("Missing Values:", X_train.isnull().sum().sum())

Training Shape: (600000, 8)
Validation Shape: (150000, 8)
Unique Fertilizers: 7
Missing Values: 0
